# [한번 해보기] 음식 리뷰 데이터 활용 유사도 검색
- corpus -> embedding vector -> 유사도 기반 검색

In [ ]:
import pandas as pd
import numpy as np

#clean_path=path.replace('#0','')
#with open(clean_path, 'r') as f:
#    data=food_df
food_df=pd.read_csv('C:\\SKN-_24\\08_LARGE_LANGUAGE_MODEL\\01_open_qi\\data\\fine_food_reviews_1k.csv')
food_df

,Unnamed: 0,Time,ProductId,UserId,Score,Summary,Text
0,0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...
...,...,...,...,...,...,...,...
995,995,1351209600,B004OQLIHK,AKHQMSUORSA91,5,Delicious!,I have ordered these raisins multiple times. ...
996,996,1351209600,B0006349W6,A21BT40VZCCYT4,5,Good Training Treat,My dog will come in from outside when I am tra...
997,997,1351209600,B00611F084,A6D4ND3C3BCYV,5,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...
998,998,1351209600,B005QKH5HA,A3LR9HCV3D96I3,5,Party Peanuts,Great product for the price. Mix with the Asia...


In [26]:
#food_df=food_df.drop(labels='Unnamed: 0', axis=1)
food_df

,Time,ProductId,UserId,Score,Summary,Text
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...
...,...,...,...,...,...,...
995,1351209600,B004OQLIHK,AKHQMSUORSA91,5,Delicious!,I have ordered these raisins multiple times. ...
996,1351209600,B0006349W6,A21BT40VZCCYT4,5,Good Training Treat,My dog will come in from outside when I am tra...
997,1351209600,B00611F084,A6D4ND3C3BCYV,5,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...
998,1351209600,B005QKH5HA,A3LR9HCV3D96I3,5,Party Peanuts,Great product for the price. Mix with the Asia...


In [ ]:
#food_df=food_df.drop(labels='Time', axis=1)
#food_df=food_df.drop(labels='ProductId', axis=1)
#food_df=food_df.drop(labels='UserId', axis=1)
food_df

,Summary,Text
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,Happy with the product,My dog was suffering with itchy skin. He had ...
...,...,...
995,Delicious!,I have ordered these raisins multiple times. ...
996,Good Training Treat,My dog will come in from outside when I am tra...
997,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...
998,Party Peanuts,Great product for the price. Mix with the Asia...


In [ ]:
# Summary,Text만 사용
# Summary (title),Text (content) --> Content
# Content 만들기 -> Embedding Vector (.csv 저장) 
# 사용자 입력 (best coffee) -> Embedding Vector2
# 유사도 계산 -> 유사도가 높은 것
# 파일을 저장하고 사용자가 입력하는 쿼리를 유사도 검사를 해보기 

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from openai import OpenAI
client=OpenAI() #객체 생성 후 간단한 질의응답을 함


response=client.chat.completions.create(
    model='gpt-4.1-mini',
    messages=[
        {'role': 'system', 'content':S'당신은 친절하고 상세한 설명을 잘하는 챗봇입니다.'},    # 높임체, 경어체 사용 시 토큰 길이가 길어짐  # 토큰 길이만으로 생각해서 작성할 때는 음, 슴체로 작성하는 것이 더 좋음
        {'role': 'user', 'title': '.'}                # user 프롬프트로 받아오기
    ],                                                                              
    temperature=1,
    max_tokens=4096,
    top_p=1
)

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'content': expected a string, got null.", 'type': 'invalid_request_error', 'param': 'messages.[1].content', 'code': None}}

In [ ]:
def text_to_embedding(texts, model='text-embedding-3-small'):
    texts=[text.replace('\n',' ') for text in text]
    response=client.embeddings.create(
        model=model,
        input=texts
    )
    return [data.embedding for data in response.data]

In [ ]:
# ebedding 모델 학습
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0
)

model.wv.vectors.shape

In [ ]:
import pandas as pd

pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

In [ ]:
model.wv.save_word2vec_format('ted_en_w2v')

In [ ]:
from gensim.models import KeyedVectors

load_model = KeyedVectors.load_word2vec_format('ted_en_w2v')

In [ ]:
load_model.most_similar('food_test')

In [ ]:
from openai import OpenAI

client=OpenAI()
summary=''
text=''

response=client.embeddings.create(
    model='text-embedding-3-small',
    input=[text]
)